# The SQL SELECT Statement

## Introduction: Asking Questions with the SELECT Statement

Now that you understand the broad capabilities of SQL and the seven fundamental operations of data retrieval, it is time to write your very first queries. Throughout this unit, we will use Microsoft's sample database, Northwind Traders, a fictitious national distributor of gourmet foods, to practice writing SQL code in a realistic business setting.<sup>\[1\]</sup>

When working with databases, the most basic task is extracting data from a table. In SQL, this is accomplished with the **SELECT** statement. 

At its simplest, a query requires two clauses:
1. **`SELECT`**: Specifies which columns you want to view.
2. **`FROM`**: Specifies the database table containing the data.

Consider the simplest possible SQL query:

```sql
SELECT *
FROM Shippers
```

This query tells SQL to retrieve all rows and all columns of the `Shippers` table. In SQL, the asterisk (`*`) is a shortcut that means "retrieve all columns." By default, SQL will return all rows. To choose specific rows, you will use the keyword `WHERE`, which you will learn in the next section of this book.

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from IPython.display import HTML, display

In [2]:
database = 'NorthwindTraders'
driver = 'ODBC Driver 18 for SQL Server'
connectionstring = f"mssql+pyodbc://(localdb)\\MSSQLLocalDB/{database}?driver={driver.replace(' ', '+')}&trusted_connection=yes"

engine = create_engine(connectionstring)

In [3]:
df = pd.read_sql('SELECT * FROM Shippers', con=engine)
display(HTML(df.to_html(index=False)))

ShipperID,CompanyName,Phone
1,Speedy Express,(503) 555-9831
2,United Package,(503) 555-3199
3,Federal Shipping,(503) 555-9931


Note that a SQL `SELECT` query **will never alter or modify** the data stored in a database. A `SELECT` statement  returns a temporary, read-only copy of the requested data.

## Selecting Specific Columns

While using `SELECT *` is convenient when exploring a table for the first time, retrieving every column is rarely necessary in professional reporting. Large enterprise tables can contain hundreds of columns. Pulling unnecessary columns wastes memory and clutters your analysis.

To retrieve specific columns, list their names after the `SELECT` keyword, separated by commas:

```sql
SELECT EmployeeID, FirstName, LastName, Title
FROM Employees
```

You can select columns in **any order you choose**, regardless of how they are physically arranged in the underlying database table. You can even request the same column multiple times if needed.

## SQL Syntax and Formatting Rules

As you write SQL queries, keep these fundamental syntax rules in mind:

* **SQL is Not Case-Sensitive:** SQL keywords (`SELECT`, `FROM`) and column names can be written in uppercase, lowercase, or mixed case. The queries `select firstname from employees` and `SELECT FirstName FROM Employees` will execute identically. However, by convention, we **capitalize SQL keywords** and use **CamelCase** for column names to make our code clean and readable.
* **Flexible Spacing and Line Breaks:** SQL ignores extra spaces, tabs, and line breaks. You can write an entire query on a single line or spread it across multiple lines.
* **No Spaces in Identifiers:** You cannot insert spaces inside a column or table name. SQL treats `LastName` and `Last Name` as distinct column names.
* **Comma Placement:** Column names in a `SELECT` clause must be separated by commas, but there must be **no comma after the final column name**.

### Using SQL's Flexibility to Improve Code Readability

SQL provides you with considerable flexibility when writing your code. You should use this flexibility to improve the readability of your code. Doing so will make it easier for yourself and others to understand and debug your code.

As an example, consider this SQL code:

```sql
select employeeid,firstname,lastname,title from employees
```

We can rewrite this code with different capitalization, spacing, and indentation to improve its readability:

```sql
SELECT 
    EmployeeID, 
    FirstName, 
    LastName, 
    Title
FROM Employees
```

Writing clean, well-formatted code is not just about aesthetics - it is a core accounting control that makes your logic easy to audit, review, and maintain.

## Thinking Like an Analyst: Four Ways to Answer "How Many Employees?"

Let's put your budding SQL skills to work and tackle a business problem. Say that the president of Northwind Traders wants to know how many employees work at the company.

In a spreadsheet, you might scroll to the bottom of a worksheet or highlight a column to check the status bar. However, in SQL, we must express our logical process explicitly. This introduces a degree of freedom that may feel new to you. There are several ways to approach this question in SQL. We will examine some different methods to show how different  assumptions about the data and the data-generating process carry different operational risks. 

### Method 1: Retrieve All Rows (`SELECT *`)

The most straightforward approach is to retrieve all rows from the `Employees` table and inspect the total record count reported by Visual Studio Code:

```sql
SELECT *
FROM Employees
```

When VS Code executes this query, the query results window displays all records and notes at the bottom: *"9 rows returned."* 


```{figure} imgs/RowCountInVSCode.png
:width: 75%
:alt: Row count in VS Code
:align: center

Row count in VS Code

While this works for small tables, retrieving millions of rows just to read a count from a software status bar is inefficient and slow.

### Method 2: Multi-Step Logic (`TOP` + `ORDER BY`)

If we know that every employee is assigned a unique, sequential `EmployeeID` starting at 1, we can sort the table in descending order by `EmployeeID` and retrieve the single highest ID:

```sql
SELECT TOP (1) EmployeeID
FROM Employees
ORDER BY EmployeeID DESC
```

This query sorts the table so the largest `EmployeeID` appears at the top, and `TOP (1)` restricts the output to that first row, returning `9`.

In [4]:
df = pd.read_sql('SELECT TOP (1) EmployeeID FROM Employees ORDER BY EmployeeID DESC', con=engine)
display(HTML(df.to_html(index=False)))

EmployeeID
9


```{warning}
**The Auditor's Perspective: Evaluating Operational Risks**  
This method relies on a dangerous assumption: that `EmployeeID` values are strictly sequential without gaps. Imagine an employee was hired and assigned ID 5. That employee later resigns and their record is deleted from the database. Another possible scenario is that the company's human resource software generates employee IDs in non-sequential blocks. In these scenarios, the maximum `EmployeeID` might be `9`, but there may only be `8` active employees in the table!

Thus, this method reports the **highest ID number**, not the **actual headcount**.
```

### Method 3: Aggregate Function (`MAX`)

Rather than sorting the entire table to find the top row, we can use SQL's built-in `MAX()` function to find the highest value in the `EmployeeID` column directly:

```sql
SELECT MAX(EmployeeID)
FROM Employees
```

This query returns a single summary result: `9`.

```{warning}
**The Auditor's Perspective: Evaluating Operational Risks**  
This method relies on the same assumption that plagues method 2. It returns the highest ID number, not the actual headcount. *The takeaway is that you must understand the company's business processes and data flows when developing your strategy for retrieving data.*
```

### Method 4: Direct Row Counting (`COUNT`)

The most direct way to count records in SQL is using the `COUNT()` function:

```sql
SELECT COUNT(*)
FROM Employees
```

This query instructs the database to count all rows in the `Employees` table, returning `9`.

Of the described methods, this one is computationally quickest and has the least risk. It simply counts the number of rows in the database table. Since there is one row per employee, it returns the number of employees in the database.

The only potential risk here is in the data itself. There is no table column indicating whether the employees are currently employed. It is possible that an employee has left the company but their information remains in the database table.

## Sorting Results with ORDER BY

By default, a database does not guarantee any specific order when returning query results. To sort your data, you must add an **`ORDER BY`** clause.

The `ORDER BY` clause syntax is:

```sql
SELECT Column1, Column2, ...
FROM TableName
ORDER BY ColumnToSort [ASC | DESC]
```

* **Ascending Order (`ASC`):** Sorts numbers from smallest to largest, dates from oldest to newest, and text alphabetically (A to Z). This is the default setting in SQL, so the `ASC` keyword is optional.
* **Descending Order (`DESC`):** Sorts numbers from largest to smallest, dates from newest to oldest, and text in reverse alphabetical order (Z to A).

### Single-Column Sort Example

Imagine the sales manager wants a list of all Northwind employees, sorted from youngest to oldest based on their birth date:

```sql
SELECT EmployeeID, FirstName, LastName, BirthDate
FROM Employees
ORDER BY BirthDate DESC
```

Because we specified `DESC`, the employee with the most recent birth date appears first.

In [5]:
df = pd.read_sql('SELECT EmployeeID, FirstName, LastName, BirthDate FROM Employees ORDER BY BirthDate DESC', con=engine)
display(HTML(df.to_html(index=False)))

EmployeeID,FirstName,LastName,BirthDate
9,Anne,Dodsworth,1966-01-27
3,Janet,Leverling,1963-08-30
6,Michael,Suyama,1963-07-02
7,Robert,King,1960-05-29
8,Laura,Callahan,1958-01-09
5,Steven,Buchanan,1955-03-04
2,Andrew,Fuller,1952-02-19
1,Nancy,Davolio,1948-12-08
4,Margaret,Peacock,1937-09-19


## Clause Sequence Rule
In SQL, clause order is strictly enforced. The `ORDER BY` clause must always appear **after** the `FROM` clause:

1. `SELECT`
2. `FROM`
3. `ORDER BY`

Attempting to place `ORDER BY` before `FROM` will result in a syntax error.

## Limiting Output with TOP

In large databases, tables often contain millions of rows. Running a query that returns millions of rows can freeze your computer and strain network bandwidth. When exploring data, you will often want to view only a small sample of records using the **`TOP`** clause.

### The `TOP` Clause Syntax

The `TOP` clause specifies the maximum number (or percentage) of rows to return:

```sql
SELECT TOP (n) ColumnList
FROM TableName
```

For example, to retrieve the first 5 records from the `Orders` table:

```sql
SELECT TOP (5) OrderID, CustomerID, OrderDate
FROM Orders
```

You can also specify a percentage of rows using `TOP (n) PERCENT`:

```sql
SELECT TOP (10) PERCENT OrderID, CustomerID, OrderDate
FROM Orders
```

### Pairing `TOP` with `ORDER BY`

Without an `ORDER BY` clause, using `TOP` returns an arbitrary set of rows based on whatever order the database happens to store them in on disk. To obtain meaningful, repeatable results - such as the top 5 highest sales transactions or the 3 most recent orders - you **must pair `TOP` with `ORDER BY`.**

Consider finding the three most expensive products sold by Northwind Traders:

```sql
SELECT TOP (3) ProductID, ProductName, UnitPrice
FROM Products
ORDER BY UnitPrice DESC
```

SQL processes this query by first sorting all products by `UnitPrice` in descending order, and then returning the top three rows from that sorted list:

In [6]:
df = pd.read_sql('SELECT TOP (3) ProductID, ProductName, UnitPrice FROM Products ORDER BY UnitPrice DESC', con=engine)
display(HTML(df.to_html(index=False)))

ProductID,ProductName,UnitPrice
38,Côte de Blaye,263.50
29,Thüringer Rostbratwurst,123.79
9,Mishi Kobe Niku,97.00


### Clause Sequence with `TOP`

Notice where `TOP` sits in the query sequence: it is placed immediately after the `SELECT` keyword, before the column list:

1. `SELECT TOP (n)`
2. Column List
3. `FROM`
4. `ORDER BY`

## Introductory Aggregations: MAX and COUNT

Throughout this course, you will frequently need to calculate summary statistics. SQL provides built-in aggregate functions that can be used directly in a `SELECT` statement to calculate single summary metrics.

### The `MAX()` Function

The `MAX()` function returns the highest value in a specified column. For example, to find the single highest freight charge in the `Orders` table:

```sql
SELECT MAX(Freight) AS HighestFreight
FROM Orders
```

Notice the use of `AS HighestFreight`. The **`AS`** keyword creates an **alias** - a descriptive custom column header for the output. If you omit `AS`, SQL will display `(No column name)` for the calculated result.

### Understanding `COUNT(*)` vs. `COUNT(column)`

The `COUNT()` function counts records, but its behavior changes depending on what is placed inside the parentheses:

* **`COUNT(*)`:** Counts **all rows** in the table, including rows that contain missing or null values in individual columns.
* **`COUNT(column_name)`:** Counts only rows where the specified column contains a **non-null (valid) value**.

Consider the `Employees` table, where some employees report to a manager (`ReportsTo`), while top executives have a `NULL` (blank) value in that field:

```sql
-- Counts all 9 employee records in the table
SELECT COUNT(*) AS TotalEmployees
FROM Employees

-- Counts only employees who have a non-null entry in the ReportsTo column
SELECT COUNT(ReportsTo) AS EmployeesWithManager
FROM Employees
```

Understanding this distinction is vital for auditors when verifying whether mandatory data fields are populated across enterprise records.

## Summary and Clause Sequence Checklist

In this chapter, you learned how to extract, format, sort, and sample tabular data using the SQL `SELECT` statement. 

### The SQL Clause Execution Sequence

Programming requires strict adherence to rules. As you build queries, memorize the required order of the SQL clauses learned so far:

```sql
SELECT TOP (n) ColumnList
FROM TableName
ORDER BY ColumnToSort [ASC | DESC]
```

| Clause | Purpose | Required or Optional? |
| :--- | :--- | :--- |
| **`SELECT`** | Specifies columns or calculations to retrieve. | **Required** |
| **`TOP (n)`** | Limits the output to $n$ rows or $n\%$ of rows. | Optional (placed immediately after `SELECT`) |
| **`FROM`** | Identifies the database source table. | **Required** |
| **`ORDER BY`** | Sorts the resulting rows in ascending or descending order. | Optional (placed after `FROM`) |

## Looking Ahead

In the next section, we will learn how to filter data tables to extract only the rows that meet specific business criteria, such as identifying overdue invoices, out-of-stock products, or sales transactions in specific geographic territories.

## Notes

\[1\] You should have received instructions on how to connect to this database in class.